In [1]:
# CELL 1: Upload the Netflix dataset CSV file to Google Colab
from google.colab import files
uploaded = files.upload()

Saving netflix_data.csv to netflix_data.csv


In [2]:
# CELL 2: Load the dataset into a pandas DataFrame and inspect it for missing values
import pandas as pd
import numpy as np

df = pd.read_csv('netflix_data.csv')

print("Original shape:", df.shape)   # Check total rows and columns
print(df.isnull().sum())             # Check how many missing values each column has

Original shape: (8807, 12)
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [3]:
# CELL 3: Fix rows where a duration value accidentally landed in the 'rating' column
# (data entry issue in the original file — these 3 rows are missing 'country',
# which caused the duration value to shift into the rating field)
shift_mask = df['rating'].isin(['74 min', '84 min', '66 min'])
df.loc[shift_mask, 'duration'] = df.loc[shift_mask, 'rating']
df.loc[shift_mask, 'rating'] = np.nan

print("Rows fixed:", shift_mask.sum())   # Should print 3

Rows fixed: 3


In [4]:
# CELL 4: Fill missing values in categorical columns and drop rows with missing dates/duration
df['director'] = df['director'].fillna('Not Specified')   # No director info available
df['cast'] = df['cast'].fillna('Not Specified')            # No cast info available
df['country'] = df['country'].fillna('Not Specified')      # No country info available
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])  # Fill with the most common rating

df = df.dropna(subset=['date_added', 'duration'])   # Drop the few rows still missing these (very small number)

print("Shape after handling missing values:", df.shape)
print(df.isnull().sum())   # Confirm no more missing values remain

Shape after handling missing values: (8797, 12)
show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [5]:
# CELL 5: Convert 'date_added' text into a proper date format and extract year/month
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y')
df['year_added'] = df['date_added'].dt.year          # Extract the year Netflix added the title
df['month_added'] = df['date_added'].dt.month_name()  # Extract the month name

print(df[['date_added', 'year_added', 'month_added']].head())   # Preview the new columns

  date_added  year_added month_added
0 2021-09-25        2021   September
1 2021-09-24        2021   September
2 2021-09-24        2021   September
3 2021-09-24        2021   September
4 2021-09-24        2021   September


In [6]:
# CELL 6: Split 'duration' into a numeric value and a unit (minutes for Movies, Seasons for TV Shows)
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(int)   # Extract just the number
df['duration_unit'] = np.where(df['type'] == 'Movie', 'min', 'Season(s)')  # Assign the correct unit

print(df[['type', 'duration', 'duration_value', 'duration_unit']].head())   # Preview the new columns

      type   duration  duration_value duration_unit
0    Movie     90 min              90           min
1  TV Show  2 Seasons               2     Season(s)
2  TV Show   1 Season               1     Season(s)
3  TV Show   1 Season               1     Season(s)
4  TV Show  2 Seasons               2     Season(s)


In [7]:
# CELL 7: Extract the first listed country and first listed genre
# (original columns can contain multiple comma-separated values, e.g. "USA, India")
df['primary_country'] = df['country'].apply(lambda x: x.split(',')[0].strip())   # Take the first country
df['primary_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0].strip())    # Take the first genre

print(df[['country', 'primary_country', 'listed_in', 'primary_genre']].head())   # Preview the new columns

         country primary_country  \
0  United States   United States   
1   South Africa    South Africa   
2  Not Specified   Not Specified   
3  Not Specified   Not Specified   
4          India           India   

                                           listed_in           primary_genre  
0                                      Documentaries           Documentaries  
1    International TV Shows, TV Dramas, TV Mysteries  International TV Shows  
2  Crime TV Shows, International TV Shows, TV Act...          Crime TV Shows  
3                             Docuseries, Reality TV              Docuseries  
4  International TV Shows, Romantic TV Shows, TV ...  International TV Shows  


In [8]:
# CELL 8: Create two extra insight-friendly columns
# 1) genre_count: how many genres are tagged to a title (shows content variety)
# 2) years_to_add: how many years after release Netflix added the title (shows catalog age/freshness)
df['genre_count'] = df['listed_in'].apply(lambda x: len(x.split(',')))

df['years_to_add'] = df['year_added'] - df['release_year']
df['years_to_add'] = df['years_to_add'].clip(lower=0)   # Prevent negative values from data errors

print(df[['listed_in', 'genre_count', 'release_year', 'year_added', 'years_to_add']].head())   # Preview

                                           listed_in  genre_count  \
0                                      Documentaries            1   
1    International TV Shows, TV Dramas, TV Mysteries            3   
2  Crime TV Shows, International TV Shows, TV Act...            3   
3                             Docuseries, Reality TV            2   
4  International TV Shows, Romantic TV Shows, TV ...            3   

   release_year  year_added  years_to_add  
0          2020        2021             1  
1          2021        2021             0  
2          2021        2021             0  
3          2021        2021             0  
4          2021        2021             0  


In [9]:
# CELL 9: Save the fully cleaned dataset as a CSV and download it to your local machine
df.to_csv('netflix_cleaned.csv', index=False)
files.download('netflix_cleaned.csv')   # This will download the file to your computer

print("Final cleaned shape:", df.shape)   # Confirm final row/column count

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Final cleaned shape: (8797, 20)


In [10]:
# CELL 10: Explore the cleaned data — this gives a first look at patterns before we move to ML
print("Content by type:\n", df['type'].value_counts())          # Movie vs TV Show count
print("\nTop 10 countries:\n", df['primary_country'].value_counts().head(10))   # Where most content is from
print("\nTop 10 genres:\n", df['primary_genre'].value_counts().head(10))        # Most common genres
print("\nRating distribution:\n", df['rating'].value_counts())                  # Age-rating breakdown
print("\nContent added by year:\n", df['year_added'].value_counts().sort_index())  # Growth trend over years

Content by type:
 type
Movie      6131
TV Show    2666
Name: count, dtype: int64

Top 10 countries:
 primary_country
United States     3205
India             1008
Not Specified      830
United Kingdom     627
Canada             271
Japan              258
France             212
South Korea        211
Spain              181
Mexico             134
Name: count, dtype: int64

Top 10 genres:
 primary_genre
Dramas                      1600
Comedies                    1210
Action & Adventure           859
Documentaries                829
International TV Shows       773
Children & Family Movies     605
Crime TV Shows               399
Kids' TV                     386
Stand-Up Comedy              334
Horror Movies                275
Name: count, dtype: int64

Rating distribution:
 rating
TV-MA       3212
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17 

In [18]:
#  Import ML libraries and convert text categories into numbers
# (Machine learning models only work with numeric data, so we encode category columns)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

le_country = LabelEncoder()
le_genre = LabelEncoder()
le_rating = LabelEncoder()
le_type = LabelEncoder()

df['country_enc'] = le_country.fit_transform(df['primary_country'])   # Encode country as numbers
df['genre_enc'] = le_genre.fit_transform(df['primary_genre'])          # Encode genre as numbers
df['rating_enc'] = le_rating.fit_transform(df['rating'])               # Encode rating as numbers
df['type_enc'] = le_type.fit_transform(df['type'])                     # Encode type (Movie/TV Show) as numbers

print(df[['primary_country', 'country_enc', 'primary_genre', 'genre_enc']].head())   # Preview the encoding

  primary_country  country_enc           primary_genre  genre_enc
0   United States           81           Documentaries         10
1    South Africa           68  International TV Shows         16
2   Not Specified           52          Crime TV Shows          8
3   Not Specified           52              Docuseries         11
4           India           30  International TV Shows         16


In [19]:
# Select the features we'll use for ML, and scale them
# (Scaling puts all numbers on the same range so no single feature dominates the model)
features = ['duration_value', 'genre_count', 'years_to_add', 'release_year',
            'country_enc', 'genre_enc', 'rating_enc']

X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # Standardize features (mean=0, std=1)

print("Features used:", features)
print("Scaled data shape:", X_scaled.shape)   # Confirm shape matches number of rows and features

Features used: ['duration_value', 'genre_count', 'years_to_add', 'release_year', 'country_enc', 'genre_enc', 'rating_enc']
Scaled data shape: (8797, 7)


In [20]:
# Group similar titles together into 5 clusters using KMeans
# (This helps us find hidden "content segments" — e.g. old classic movies, short kids content, etc.)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)   # Assign each title to one of 5 clusters

print(df['cluster'].value_counts().sort_index())   # Check how many titles fall into each cluster

cluster
0    2080
1    1530
2    2228
3     515
4    2444
Name: count, dtype: int64


In [21]:
# Build a summary profile for each cluster
# (This tells us the "personality" of each content segment — average duration, genre variety, etc.)
cluster_profile = df.groupby('cluster')[['duration_value', 'genre_count',
                                          'years_to_add', 'release_year']].mean().round(1)

cluster_profile['count'] = df['cluster'].value_counts().sort_index()
cluster_profile['top_genre'] = df.groupby('cluster')['primary_genre'].agg(lambda x: x.value_counts().index[0])
cluster_profile['top_country'] = df.groupby('cluster')['primary_country'].agg(lambda x: x.value_counts().index[0])
cluster_profile['dominant_type'] = df.groupby('cluster')['type'].agg(lambda x: x.value_counts().index[0])

print(cluster_profile)   # Print the full profile table for all 5 clusters

         duration_value  genre_count  years_to_add  release_year  count  \
cluster                                                                   
0                   5.3          2.7           1.8        2017.1   2080   
1                  37.5          1.1           2.0        2016.7   1530   
2                  99.1          2.0           4.1        2014.8   2228   
3                 109.2          2.3          33.0        1986.0    515   
4                 110.3          2.6           3.4        2015.5   2444   

                      top_genre    top_country dominant_type  
cluster                                                       
0        International TV Shows  United States       TV Show  
1               Stand-Up Comedy  United States         Movie  
2                        Dramas  United States         Movie  
3            Action & Adventure  United States         Movie  
4                        Dramas          India         Movie  


In [22]:
# Train a Random Forest model to predict Movie vs TV Show
# (The goal here isn't the prediction itself, but to see which features matter most for content type)
X_clf = df[['duration_value', 'genre_count', 'years_to_add', 'release_year',
            'country_enc', 'genre_enc', 'rating_enc']]
y_clf = df['type_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)   # Split data into 80% training, 20% testing

rf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10)
rf.fit(X_train, y_train)   # Train the model

y_pred = rf.predict(X_test)   # Predict on unseen test data

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, target_names=le_type.classes_))

Accuracy: 0.999
              precision    recall  f1-score   support

       Movie       1.00      1.00      1.00      1227
     TV Show       1.00      1.00      1.00       533

    accuracy                           1.00      1760
   macro avg       1.00      1.00      1.00      1760
weighted avg       1.00      1.00      1.00      1760



In [23]:
# Check which features the model relied on most
# (This is the real insight — it tells us what actually differentiates Movies from TV Shows)
importance = pd.Series(rf.feature_importances_, index=X_clf.columns).sort_values(ascending=False)

print("Feature importance (higher = more influence on prediction):")
print(importance.round(3))

Feature importance (higher = more influence on prediction):
duration_value    0.761
genre_enc         0.168
rating_enc        0.030
years_to_add      0.012
genre_count       0.011
country_enc       0.011
release_year      0.008
dtype: float64


In [24]:
# Save the final dataset (with cluster labels) and the insight tables as CSV files
# (These files will be loaded into MySQL in Step 3)
df.to_csv('netflix_with_ml.csv', index=False)                              # Full data + cluster labels
cluster_profile.to_csv('cluster_profile.csv')                              # Cluster summary table
importance.to_frame('importance').to_csv('feature_importance.csv')        # Feature importance table

files.download('netflix_with_ml.csv')
files.download('cluster_profile.csv')
files.download('feature_importance.csv')

print("All 3 files saved and downloaded successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All 3 files saved and downloaded successfully!
